![image_1779902925022.png](./image_1779902925022.png "image_1779902925022.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Initialize Spark
spark = SparkSession.builder.appName("StocksDF").getOrCreate()

# Define schema
schema = StructType([
    StructField("stock_name", StringType(), True),
    StructField("operation", StringType(), True),
    StructField("operation_day", IntegerType(), True),
    StructField("price", IntegerType(), True)
])

# Data
data = [
    ("Leetcode", "Buy", 1, 1000),
    ("Corona", "Buy", 2, 3000),
    ("Corona", "Sell", 3, 1200),
    ("Leetcode", "Sell", 5, 9000),
    ("Leetcode", "Buy", 6, 4000),
    ("Leetcode", "Sell", 8, 5000)
]

# Create DataFrame
df = spark.createDataFrame(data, schema=schema)

# Show DataFrame
df.show()

In [0]:

from pyspark.sql import functions as f
from pyspark.sql.window import Window

window_spec = Window.partitionBy("stock_name")
df_result=(
    df.withColumn(
        "sum_of_all_sell_prices",
        f.sum(
            f.when(f.col("operation") == "Sell", f.col("price"))
            ).over(window_spec)
        )
    .withColumn(
        "sum_of_all_buy_prices",
        f.sum(
            f.when(f.col("operation") == "Buy", f.col("price"))
        ).over(window_spec)
    )
    .withColumn(
        "capital_gain_loss",
        f.col("sum_of_all_sell_prices") - f.col("sum_of_all_buy_prices")
    )
    .select(
        "stock_name",
        "capital_gain_loss"
    )
    .distinct()
)
# print(df_result)
df_result.show()